In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split , GridSearchCV
from sklearn.preprocessing import StandardScaler , OneHotEncoder , LabelEncoder
from scikeras.wrappers import KerasClassifier
from keras.models import Sequential
from keras.layers import Dense
from keras.callbacks import EarlyStopping , TensorBoard
import pickle

In [9]:
data = pd.read_csv('./training_data/Churn_Modelling.csv')

print(data.head())

   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1               1   

   EstimatedSalary  Exited  
0         90437.74       1  
1         81341.31       0  
2        112912.73       1  
3         95174.52       0  
4        1149

In [10]:
import numpy as np

data = data.drop(['RowNumber','CustomerId','Surname'],axis=1)

label_encoder_gender = LabelEncoder()
data['Gender'] = np.asarray(label_encoder_gender.fit_transform(data['Gender']))

print(data.head())

   CreditScore Geography  Gender  Age  Tenure    Balance  NumOfProducts  \
0          619    France       0   42       2       0.00              1   
1          608     Spain       0   41       1   83807.86              1   
2          502    France       0   42       8  159660.80              3   
3          699    France       0   39       1       0.00              2   
4          850     Spain       0   43       2  125510.82              1   

   HasCrCard  IsActiveMember  EstimatedSalary  Exited  
0          1               1         90437.74       1  
1          0               1         81341.31       0  
2          1               0        112912.73       1  
3          0               0         95174.52       0  
4          1               1        114968.34       0  


In [7]:
onehot_encoder_geo = OneHotEncoder(sparse_output=False)

geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']])

geo_encoded_title = onehot_encoder_geo.get_feature_names_out(['Geography'])

geo_encoded_df = pd.DataFrame(geo_encoded,columns=geo_encoded_title)

new_data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

print(new_data.head())


   CreditScore  Gender  Age  Tenure    Balance  NumOfProducts  HasCrCard  \
0          619       0   42       2       0.00              1          1   
1          608       0   41       1   83807.86              1          0   
2          502       0   42       8  159660.80              3          1   
3          699       0   39       1       0.00              2          0   
4          850       0   43       2  125510.82              1          1   

   IsActiveMember  EstimatedSalary  Exited  Geography_France  \
0               1         90437.74       1               1.0   
1               1         81341.31       0               0.0   
2               0        112912.73       1               1.0   
3               0         95174.52       0               1.0   
4               1        114968.34       0               0.0   

   Geography_Germany  Geography_Spain  
0                0.0              0.0  
1                0.0              1.0  
2                0.0              0.0 

In [11]:
X = new_data.drop('Exited',axis=1)
Y = new_data['Exited']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [12]:
# Save encoders and scaler for later use
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [13]:
## Define a function to create the model and try different parameters(KerasClassifier)

from keras import Input , Sequential
from keras.optimizers import Adam
from keras.losses import BinaryCrossentropy

def create_model(neurons=32,layers=1):
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],)))
    model.add(Dense(units=neurons,activation='relu'))

    for _ in range(layers-1):
        model.add(Dense(units=neurons,activation='relu'))

    model.add(Dense(units=1,activation='sigmoid'))

    opt = Adam(learning_rate=0.01)
    loss = BinaryCrossentropy()

    model.compile(optimizer=opt,loss=loss,metrics=['accuracy'])

    return model

    

In [18]:
## Create a Keras classifier
model = KerasClassifier(model=create_model,verbose=1,neurons=32,layers=1)

In [20]:
# Define the grid search parameters
param_grid = {
    'neurons': [16, 32, 64, 128],
    'layers': [1, 2],
    'epochs': [50, 100]
}

In [21]:
# Perform grid search
grid = GridSearchCV(estimator=model,param_grid=param_grid,cv=3,verbose=1,n_jobs=1)
grid_result = grid.fit(X_train,Y_train)

# Print the best parameters
print(f"Best : {grid_result.best_score_} using {grid_result.best_params_}")



Fitting 3 folds for each of 16 candidates, totalling 48 fits
Epoch 1/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 295us/step - accuracy: 0.8046 - loss: 0.4369  
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 249us/step - accuracy: 0.8451 - loss: 0.3726
Epoch 3/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 251us/step - accuracy: 0.8479 - loss: 0.3602
Epoch 4/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 257us/step - accuracy: 0.8519 - loss: 0.3530
Epoch 5/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 252us/step - accuracy: 0.8551 - loss: 0.3524
Epoch 6/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 255us/step - accuracy: 0.8551 - loss: 0.3477
Epoch 7/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 262us/step - accuracy: 0.8547 - loss: 0.3483
Epoch 8/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 255us/step - accuracy: 0.8573 - loss: 0.3480
Epoch 9/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 253us/step - accuracy: 0.8581 - loss: 0.3452
Epoch 10/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 258us/step - accuracy: 0.8577 - loss: 0.3427
Epoch 11/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 348us/step -